:important: This is the notebook for my Wildfires project for SDS210. Broad structure as follows:

1. Import required packages
    

2. test pull data into project with the FIRMS API and then just use the bounding box for Australia.

3. reproject to correct CRS


4. 

In [1]:
import requests
import pandas as pd
import geopandas as gpd
import time
import folium

In [2]:
# We need to access the API and to do that, will use the map key that permits access.
MAP_KEY = '54684dde74a099b139ddbbef0f621891'

# Now let's check how many results we have

url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

transaction_limit             5000
current_transactions           254
transaction_interval    10 minutes
dtype: object

In [3]:
# let's create a simple function that tells us how many transactions we have used.
# We will use this in later examples

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

tcount = get_transaction_count()
print ('Our current transaction count is %i' % tcount)

Our current transaction count is 254


In [4]:
# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
df = pd.read_csv(da_url)
display(df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-02-01,2026-05-11
1,MODIS_SP,2000-11-01,2026-01-31
2,VIIRS_NOAA20_NRT,2026-03-01,2026-05-11
3,VIIRS_NOAA20_SP,2018-04-01,2026-02-28
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-11
5,VIIRS_SNPP_NRT,2026-03-01,2026-05-11
6,VIIRS_SNPP_SP,2012-01-20,2026-02-28
7,LANDSAT_NRT,2022-06-20,2026-05-10
8,GOES_NRT,2022-08-09,2026-05-11
9,BA_MODIS,2000-11-01,2026-02-01


In [5]:
# now let's see how many transactions we use by querying this end point

start_count = get_transaction_count()
pd.read_csv(da_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

# now remember, after 10 minutes this will reset


We used 5 transactions.


In [6]:
# in this example let's look at VIIRS NOAA-20, entire world and the most recent day
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'
start_count = get_transaction_count()
df_area = pd.read_csv(area_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

df_area

We used 36 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,19.40142,-155.28870,346.90,0.42,0.45,2026-05-11,1,N20,VIIRS,l,2.0NRT,317.25,4.43,D
1,19.40241,-155.28065,350.06,0.42,0.45,2026-05-11,1,N20,VIIRS,l,2.0NRT,328.95,13.71,D
2,19.40290,-155.27666,367.00,0.42,0.45,2026-05-11,1,N20,VIIRS,h,2.0NRT,340.95,21.64,D
3,19.40338,-155.27271,367.00,0.42,0.45,2026-05-11,1,N20,VIIRS,h,2.0NRT,341.46,21.64,D
4,19.40387,-155.26875,353.63,0.42,0.45,2026-05-11,1,N20,VIIRS,n,2.0NRT,327.84,17.25,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19043,32.13406,6.47055,356.22,0.57,0.69,2026-05-11,1333,N20,VIIRS,n,2.0NRT,312.22,6.01,D
19044,32.13575,6.47003,354.17,0.57,0.69,2026-05-11,1333,N20,VIIRS,n,2.0NRT,312.03,14.25,D
19045,32.51077,6.71272,342.54,0.59,0.70,2026-05-11,1333,N20,VIIRS,n,2.0NRT,310.07,6.85,D
19046,32.55280,3.17126,347.26,0.37,0.58,2026-05-11,1333,N20,VIIRS,n,2.0NRT,311.03,4.39,D


In [7]:
# We are particularly interested in the wildfires in Australia and so will select this information using a bounding box. Aus = 110 -55, 180 -10
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/110,-50,170,-12/3'
df_area = pd.read_csv(area_url)
df_area

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,-16.61034,137.37132,306.02,3.64,1.79,2026-05-09,9,Terra,MODIS,54,6.1NRT,291.94,24.95,D
1,-16.14329,132.06619,307.23,1.42,1.18,2026-05-09,9,Terra,MODIS,30,6.1NRT,293.96,7.34,D
2,-16.14172,132.05327,308.09,1.41,1.18,2026-05-09,9,Terra,MODIS,37,6.1NRT,294.15,7.87,D
3,-16.13570,132.06056,314.82,1.41,1.18,2026-05-09,9,Terra,MODIS,72,6.1NRT,295.73,15.11,D
4,-16.13390,131.98894,314.07,1.40,1.17,2026-05-09,9,Terra,MODIS,72,6.1NRT,294.78,14.81,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1339,-34.24023,115.17585,301.41,1.00,1.00,2026-05-11,1336,Terra,MODIS,41,6.1NRT,283.41,6.91,N
1340,-32.59797,115.98985,319.61,1.06,1.03,2026-05-11,1336,Terra,MODIS,99,6.1NRT,269.96,23.54,N
1341,-32.59631,116.00082,324.62,1.06,1.03,2026-05-11,1336,Terra,MODIS,100,6.1NRT,269.87,29.15,N
1342,-32.41375,118.35549,302.73,1.34,1.15,2026-05-11,1336,Terra,MODIS,51,6.1NRT,284.12,12.36,N


In [8]:
# Convert to GeoDataFrame (WGS84)
fires_gdf = gpd.GeoDataFrame(
    df_area, 
    geometry=gpd.points_from_xy(
        df_area["longitude"], 
        df_area["latitude"]
    ),
    crs="EPSG:4326")

print(f"Found {len(fires_gdf)} fire records.")

Found 1344 fire records.


In [9]:
# Initialise a map centered on Australia
australia_map = folium.Map(
    location=[-28.281828, 136.145401],
    zoom_start=5,
    tiles="CyclOSM",  # A clean, light basemap
)

# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(fires_gdf).add_to(australia_map)

australia_map
